# Generate TFLite Model for ESP32 HIL Deployment

This notebook converts the ONNX model to TFLite format and generates the C header file (`model_data.h`) required for ESP32 deployment.

**IMPORTANT REQUIREMENTS**:
- Must run in Google Colab with Python 3.10-3.11 (default Colab runtime)
- Will NOT work on Python 3.12+ due to TensorFlow compatibility
- Local Python 3.14 cannot install TensorFlow

**If you encounter errors**:
1. Check Python version: `!python --version` (should be 3.10.x)
2. If using Colab, ensure you're on default runtime (not custom)
3. For local testing, use `test_onnx_model.py` (works on Python 3.14)

## 1. Check Python Version

In [ ]:
!python --version

## 2. Install Dependencies

In [ ]:
!pip install -q onnx onnx-tf tensorflow
print("Dependencies installed!")

import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_saved_model("tf_model")
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS]
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

tflite_path = "sensorfusion_esp32_v2.tflite"
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)
print(f"TFLite model: {tflite_path}")

In [ ]:
import onnx
from onnx2tf import onnx2tf

onnx_model = onnx.load(onnx_files[0])
print("Converting to TFLite...")
tflite_model, _ = onnx2tf(onnx_model=onnx_model, output_folder_path="tflite_output", non_verbose=True)
print("Conversion complete!")

In [ ]:
import glob
tflite_path = glob.glob("tflite_output/**/*.tflite", recursive=True)[0]
print(f"TFLite model: {tflite_path}")

## 4. Validate TFLite

In [ ]:
import tensorflow as tf
import numpy as np

interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print(f"Input: {input_details[0]['shape']}, {input_details[0]['dtype']}")
print(f"Output: {output_details[0]['shape']}, {output_details[0]['dtype']}")

In [ ]:
test_input = np.random.randn(1, 50, 6).astype(np.float32)
interpreter.set_tensor(input_details[0]['index'], test_input)
interpreter.invoke()
output = interpreter.get_tensor(output_details[0]['index'])
print(f"Inference test passed! Output shape: {output.shape}")

## 5. Generate C Header

In [ ]:
with open(tflite_path, 'rb') as f:
    model_bytes = f.read()

hex_array = ', '.join([f'0x{b:02x}' for b in model_bytes])
array_lines = [hex_array[i:i+80] for i in range(0, len(hex_array), 80)]

header_content = f"""/*
 * TFLite model for SensorFusion-HAR ESP32
 * Size: {len(model_bytes)} bytes
 */

#ifndef MODEL_DATA_H
#define MODEL_DATA_H

#include <stdint.h>

alignas(8) const unsigned char model_tflite[] = {{
{chr(10).join(['    ' + line for line in array_lines])}
}};

const unsigned int model_tflite_len = sizeof(model_tflite);

#endif
"""

with open("model_data.h", 'w') as f:
    f.write(header_content)

print(f"model_data.h generated ({len(model_bytes)} bytes)")

## 6. Download

Download `model_data.h` and place in `HIL_Implementation/esp32/`